# ModeTrends — Requêtez des services IA

Adaptation du notebook OpenClassrooms `huggingface_api_cloth_seg.ipynb`.
Modèle imposé : `sayeed99/segformer_b3_clothes`.

**État vérifié le 20 octobre 2026 :** authentification réussie, première inférence
refusée avec HTTP 402 (crédits insuffisants). Les appels ont ensuite été arrêtés
à la demande de l'utilisateur. Les cellules ci-dessous fonctionnent hors réseau
par défaut. Aucune mesure de qualité n'est inventée.

Les fonctions sont regroupées dans `modetrends.py` pour éviter de maintenir
deux implémentations divergentes. Leur code contient les explications détaillées.

## 1. Configuration initiale et importations
Installer les dépendances selon le README, puis sélectionner le kernel de l'environnement virtuel.

In [1]:
from pathlib import Path
import json
import os
import sys
from IPython.display import display, Markdown, Image as DisplayImage
from dotenv import load_dotenv

# Fonctionne si le notebook est lancé depuis le dossier du projet.
ROOT = Path.cwd()
if not (ROOT / "modetrends.py").exists():
    raise RuntimeError("Ouvrir le notebook depuis le dossier modetrends.")
sys.path.insert(0, str(ROOT))
import modetrends as mt
load_dotenv(ROOT / ".env", encoding="utf-8-sig")
print("Token configuré :", bool(os.getenv("HF_TOKEN")))
# Ne jamais afficher la valeur du token.


Token configuré : True


## 2. Comprendre l'API et contrôler les données
Le routeur actuel remplace l'adresse historique du notebook fourni.

In [2]:
print("Modèle :", mt.MODEL)
print("Endpoint :", mt.API_URL)
pairs = mt.dataset_pairs()
splits = mt.make_splits(pairs)
print("Paires validées :", len(pairs))
print("Répartition :", {name: len(ids) for name, ids in splits.items()})
print("Classes :", mt.CLASS_MAPPING)


Modèle : sayeed99/segformer_b3_clothes
Endpoint : https://router.huggingface.co/hf-inference/models/sayeed99/segformer_b3_clothes
Paires validées : 50
Répartition : {'train': 30, 'validation': 10, 'test': 10}
Classes : {'Background': 0, 'Hat': 1, 'Hair': 2, 'Sunglasses': 3, 'Upper-clothes': 4, 'Skirt': 5, 'Pants': 6, 'Dress': 7, 'Belt': 8, 'Left-shoe': 9, 'Right-shoe': 10, 'Face': 11, 'Left-leg': 12, 'Right-leg': 13, 'Left-arm': 14, 'Right-arm': 15, 'Bag': 16, 'Scarf': 17}


## 3. Préparation des masques et métriques

Les masques API sont des PNG binaires encodés en base64. `create_masks` associe
leurs noms aux 18 identifiants de classe, conserve les contours au plus proche
voisin et empêche le fond d'effacer un vêtement. En cas de chevauchement, le score
du segment départage les classes.

La mIoU moyenne l'intersection / union des classes présentes. Le Dice vaut
2 × intersection / somme des surfaces. Les calculs partent de matrices de
confusion cumulées, avec une analyse spécifique des vêtements et accessoires.
Une classe absente des deux masques ne reçoit pas artificiellement 100 %.

In [3]:
# Vérification mathématique sur un cas SYNTHÉTIQUE, pas un résultat du modèle.
import numpy as np
truth = np.array([[4, 4], [6, 6]])
prediction = np.array([[4, 6], [6, 6]])
check = mt.metrics(mt.confusion_matrix(truth, prediction))
print("mIoU synthétique attendue 7/12 :", check["miou"])
assert abs(check["miou"] - 7/12) < 1e-12


mIoU synthétique attendue 7/12 : 0.5833333333333333


## 4. Segmentation d'une image

Activer la variable ci-dessous **uniquement après ajout de crédits** et pour
demander explicitement un appel API. Le secret reste dans `.env`.
Les réponses réussies sont mises en cache, ce qui permet une reprise du lot.
Une erreur de crédit interrompt immédiatement l'exécution.

In [4]:
EXECUTER_API = False  # Mettre True pour autoriser les appels réels.
if EXECUTER_API:
    mt.run(limit=1)
else:
    print("Appels API désactivés. Dernier essai réel : HTTP 402.")


Appels API désactivés. Dernier essai réel : HTTP 402.


## 5. Traitement des 50 images
Les succès déjà présents sont réutilisés. Aucun résultat synthétique n'est injecté dans l'évaluation.

In [5]:
if EXECUTER_API:
    mt.run(limit=50)
else:
    print("Lot de 50 images prêt, non exécuté faute de crédits.")


Lot de 50 images prêt, non exécuté faute de crédits.


## 6. Visualisations et bilan
Après succès, chaque image dispose de quatre vues : original, annotation, prédiction et superposition.

In [6]:
result_dir = ROOT / "results"
if (result_dir / "run.json").exists():
    records = json.loads((result_dir / "run.json").read_text("utf-8"))
    summary = mt.report(result_dir, records, splits)
    print("Couverture constatée :", summary["coverage"])
    print("mIoU du lot :", summary["all"]["miou"])
    for path in sorted((result_dir / "figures").glob("*.png"))[:3]:
        display(DisplayImage(filename=str(path)))
    display(Markdown((result_dir / "rapport.md").read_text("utf-8")))


Couverture constatée : {'requested': 1, 'successful': 0, 'failed': 1}
mIoU du lot : None


# Bilan ModeTrends

Modèle : `sayeed99/segformer_b3_clothes`. Exécution via HF Inference.
Couverture : 0 succès sur 1 images demandées.
Objectif complet : 50 images. Une tentative refusée ne constitue pas une segmentation.

| Population | mIoU toutes classes | mIoU vêtements/accessoires | Dice macro |
|---|---:|---:|---:|
| Lot exploratoire | non mesuré | non mesuré | non mesuré |
| Train réservé | non mesuré | non mesuré | non mesuré |
| Validation | non mesuré | non mesuré | non mesuré |
| Test réservé | non mesuré | non mesuré | non mesuré |

## Interprétation et protocole

IoU = intersection / union. Dice = 2 × intersection / somme des surfaces.
Les matrices de confusion sont cumulées pixel par pixel avant la moyenne
des classes. Les classes absentes de la référence ET de la prédiction sont
exclues de la moyenne, et restent null dans le JSON.
Le fond peut gonfler l'accuracy : la mIoU vêtements est présentée séparément.
Les échecs API sont comptés dans la couverture, jamais transformés en prédictions.

Les 50 paires ont été réparties à graine fixe 42 : 30 train, 10 validation,
10 test. Aucun réentraînement n'est effectué. Les 30 images « train » ne
servent que de réserve pour une éventuelle adaptation future. Toute décision
de réglage doit être prise sur validation avant un test final.
Le bilan global est exploratoire. Le jeu fourni provient de human_parsing_dataset
(ATR), également cité comme source d'entraînement du modèle : le recouvrement
avec ses données d'entraînement n'est pas exclu. Ces scores ne prouvent donc
pas une généralisation sur de nouvelles publications Instagram. Il faut un
test indépendant, séparé par personne et par publication, avant production.
Les classes couvrent vêtements, accessoires et parties du corps. Il s'agit
de segmentation sémantique : deux personnes portant la même classe ne sont
pas distinguées comme instances.

## Coût pour 500 000 images / 30 jours

Volume moyen : 16 667 images/jour, soit 0,193 image/seconde.
HF Inference facture le temps de calcul multiplié par le tarif du matériel.
Le tarif unitaire de ce déploiement n'est pas publié dans les sources consultées.
La latence réseau mesurée ne permet pas de déduire la facture.

Scénarios de sensibilité à 0,50 USD/heure **supposé**, avec 5 % d'appels
supplémentaires facturés : 0,5 s/image → 36,46 USD ; 2 s → 145,83 USD ;
5 s → 364,58 USD. Ce sont des hypothèses, pas un devis Hugging Face.
Un endpoint dédié actif 720 heures au même tarif supposé coûterait 360 USD,
mais sa capacité et sa tarification doivent être vérifiées séparément.
Ces montants excluent taxes, stockage et crédits éventuels.

Pour une estimation fiable : relever le coût d'un lot suffisamment grand dans
la facturation HF, puis calculer 500 000 × coût du lot / nombre de succès.
Les erreurs et réessais du lot seront ainsi inclus. Ne pas multiplier encore
par les réessais si le coût du lot les contient déjà.

Sources consultées le 20/10/2026 :
- https://huggingface.co/docs/inference-providers/pricing
- https://huggingface.co/docs/inference-providers/tasks/image-segmentation
- https://huggingface.co/sayeed99/segformer_b3_clothes


## 7. Chiffrage de 500 000 images sur 30 jours

Le tarif exact du déploiement et son temps de calcul facturé ne sont pas connus.
Les scénarios suivants utilisent un tarif **supposé** de 0,50 USD/heure et 5 %
de tentatives supplémentaires. La durée HTTP ne remplace pas la durée facturée.
L'estimation devra être recalibrée sur le coût réel d'un lot dans la facturation.

In [7]:
for seconds in [0.5, 2, 5]:
    scenario = mt.cost_scenarios(seconds=seconds)
    print(f"{seconds:g} s/image : {scenario['estimated_usd']:.2f} USD (hypothèse)")


0.5 s/image : 36.46 USD (hypothèse)
2 s/image : 145.83 USD (hypothèse)
5 s/image : 364.58 USD (hypothèse)


## Conclusion et prochaines étapes

Le logiciel est testé hors réseau et le refus de crédit a été observé en réel.
Pour conclure sur le modèle : exécuter les 50 images, examiner les erreurs par
classe et les cas difficiles, puis compléter la présentation avec les mesures.
Le jeu ATR peut recouper l'entraînement du modèle. Un test indépendant de
nouvelles publications, regroupées par personne, reste indispensable.

Sources : [modèle](https://huggingface.co/sayeed99/segformer_b3_clothes),
[API](https://huggingface.co/docs/inference-providers/tasks/image-segmentation),
[tarification](https://huggingface.co/docs/inference-providers/pricing).